---
title: "DRG Checks"

author: "Carlos Resurreccion"

date: "2025-04-03"
---


# Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in
`~/pids-drg-claims/data-cleaning/00a-parameters.r`

Change seldom touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`


In [ ]:
source(here::here("data-cleaning", "00a-parameters.r"))


# Libraries


In [ ]:
source(here::here("data-cleaning", "00b-packages.r"))


# R Scripts


In [ ]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


# Load Mapping Data


In [ ]:
source(here::here("data-cleaning", "00d-load-mapping.r"))


# Data Verification Proper


## Load final .rds


In [ ]:
dt <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_b_bq_subset", ".rds"
  )
))


## Function Definitions

In [ ]:
test_checks <- function(section_id) {
  # Coerce to two-character string (e.g., 1 → "01")
  section_id <- sprintf("%02d", as.integer(section_id))

  # Get the calling environment (e.g., global or wherever this is invoked from)
  calling_env <- parent.frame()

  # Build pattern to match only variables for the given section
  pattern <- paste0("^chk_", section_id, "_\\d{2}_.+")

  # List relevant check variables in the calling environment
  chk_vars <- ls(envir = calling_env, pattern = pattern)

  # If no matching checks found, warn and exit
  if (length(chk_vars) == 0) {
    message("⚠️ No checks found for section: ", section_id)
    return(invisible(NULL))
  }

  # Get values (assumed format: c(flag, info))
  chk_values_raw <- lapply(chk_vars, get, envir = calling_env)

  # Extract just the logical flag from each
  chk_flags <- sapply(chk_values_raw, function(x) isTRUE(x[1]))

  # Check for failures
  if (any(!chk_flags)) {
    failed_checks <- chk_vars[!chk_flags]

    # Build detailed failure messages
    failure_messages <- mapply(function(var, val) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", var)
      info <- if (length(val) > 1) val[2] else "No additional info"
      paste0("• ", suffix, ": ", info)
    }, var = failed_checks, val = chk_values_raw[!chk_flags], SIMPLIFY = TRUE)

    stop(paste0(
      "❌ Validation failed in section ", section_id, ":\n",
      paste(failure_messages, collapse = "\n")
    ))
  } else {
    # Print all check results
    cat(paste0("✅ All validation checks passed for section ", section_id, ":\n"))
    for (i in seq_along(chk_vars)) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", chk_vars[i])
      cat(paste0(suffix, ": ", chk_values_raw[[i]][1], "\n"))
    }
  }
}

debug_setequal <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    # Identify elements missing and extra
    missing_in_y <- setdiff(x, y) # present in x but missing in y
    extra_in_y <- setdiff(y, x) # present in y but not in x

    # Compose detailed message
    mismatch_msg <- paste0(
      if (length(missing_in_y)) {
        paste0("\n  - missing: ", paste(missing_in_y, collapse = ", "))
      } else {
        ""
      },
      if (length(extra_in_y)) {
        paste0("\n  - invalid: ", paste(extra_in_y, collapse = ", "))
      } else {
        ""
      }
    )

    return(c(FALSE, mismatch_msg))
  }
}

## Test Batch 01:

In [ ]:
cols_expected <- bq_cols
cols_actual <- colnames(dt)
cols_schema <- fromJSON(here(
  "data-cleaning/r_scripts_v2",
  "bq_schema_cleaning.json"
))$name
chk_01_01_cols_match_expected <- debug_setequal(cols_expected, cols_actual)
chk_01_02_cols_match_schema <- debug_setequal(cols_schema, cols_actual)
test_checks(1)


✅ All validation checks passed for section 01:
cols_match_expected: TRUE
cols_match_schema: TRUE
